# Aula 5 - Agents

## Versão 5.0 do projeto: um assistente que decide os próximos passos

Na Aula 3, a aplicação fazia no máximo um ciclo: o modelo pedia uma ferramenta, a aplicação executava e o modelo respondia. Mas algumas perguntas exigem mais de um passo, e o segundo passo depende do resultado do primeiro. Nesta aula, vamos colocar esse ciclo dentro de um loop de decisão.

Ao final, você deve conseguir:

- explicar a diferença entre uma chamada com tool e um agente;
- executar ferramentas de forma segura e devolver erros ao modelo como resultado;
- implementar um loop que só termina quando o modelo responde sem pedir ferramentas;
- limitar o número de iterações para evitar loops infinitos;
- registrar cada passo para entender e depurar o comportamento do agente.

## 1. Configuração do ambiente

Instale as dependências pelo terminal, seguindo o `README.md` desta pasta. Copie `env.example` para `.env` e preencha `GROQ_API_KEY`. Nunca coloque a chave diretamente neste arquivo nem faça commit do `.env`.

In [ ]:
import json
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    raise RuntimeError(
        "GROQ_API_KEY não encontrada. Copie env.example para .env e informe sua chave."
    )

client = OpenAI(
    api_key=api_key,
    base_url=os.getenv("GROQ_BASE_URL", "https://api.groq.com/openai/v1"),
)

MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-120b")
print(f"Modelo configurado: {MODEL}")

## 2. Ferramentas do agente

Desta vez, as funções e os schemas já estão prontos, porque o foco é o loop. Repare que agora existem ferramentas cujos resultados alimentam outras: `consultar_notas` devolve a lista que `calcular_media` precisa, e `listar_aulas` mostra quais números `consultar_aula` aceita.

Se na Aula 4 você expôs ferramentas por um MCP server, elas podem substituir estas. O loop é o mesmo.

In [ ]:
STUDY_CONTENT = {
    "1": {"tema": "LLM APIs", "resumo": "Chamadas, mensagens, tokens e ausência de memória automática."},
    "2": {"tema": "Context Engineering", "resumo": "Seleção e organização do contexto antes da chamada."},
    "3": {"tema": "Tool Calling", "resumo": "Contratos de ferramentas, pedidos estruturados e execução pela aplicação."},
    "4": {"tema": "MCP + SDKs", "resumo": "Ferramentas padronizadas e expostas por servidores MCP."},
    "5": {"tema": "Agents", "resumo": "Loop de decisão que repete chamadas e ferramentas até a resposta final."},
    "6": {"tema": "RAG + Fechamento", "resumo": "Recuperação de documentos antes de gerar a resposta."},
}

NOTAS = {
    "algoritmos": [7.5, 6.0, 8.0],
    "banco_de_dados": [5.0, 4.5, 6.0],
}


def listar_aulas():
    return {"aulas": [{"aula": numero, "tema": item["tema"]} for numero, item in STUDY_CONTENT.items()]}


def consultar_aula(aula):
    numero = str(aula)
    item = STUDY_CONTENT.get(numero)
    if item is None:
        return {"found": False, "aula": numero}
    return {"found": True, "aula": numero, **item}


def consultar_notas(disciplina):
    chave = str(disciplina).strip().lower()
    if chave not in NOTAS:
        return {"found": False, "disciplina": chave, "disciplinas_disponiveis": sorted(NOTAS)}
    return {"found": True, "disciplina": chave, "notas": NOTAS[chave]}


def calcular_media(notas):
    if not isinstance(notas, list) or not notas:
        raise ValueError("notas deve ser uma lista não vazia")
    if not all(isinstance(n, (int, float)) and 0 <= n <= 10 for n in notas):
        raise ValueError("cada nota deve ser um número entre 0 e 10")
    media = sum(notas) / len(notas)
    return {"media": round(media, 2), "quantidade": len(notas), "aprovado": media >= 6}


AVAILABLE_TOOLS = {
    "listar_aulas": listar_aulas,
    "consultar_aula": consultar_aula,
    "consultar_notas": consultar_notas,
    "calcular_media": calcular_media,
}

tools = [
    {
        "type": "function",
        "function": {
            "name": "listar_aulas",
            "description": "Lista os números e temas de todas as aulas da oficina. Use para descobrir qual aula trata de um assunto.",
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "consultar_aula",
            "description": "Retorna o tema e o resumo de uma aula a partir do número dela.",
            "parameters": {
                "type": "object",
                "properties": {"aula": {"type": "string", "description": "Número da aula, por exemplo '3'."}},
                "required": ["aula"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "consultar_notas",
            "description": "Retorna a lista de notas do aluno em uma disciplina.",
            "parameters": {
                "type": "object",
                "properties": {"disciplina": {"type": "string", "description": "Nome da disciplina, por exemplo 'algoritmos'."}},
                "required": ["disciplina"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calcular_media",
            "description": "Calcula a média de uma lista de notas entre 0 e 10 e informa se o aluno foi aprovado (média mínima 6).",
            "parameters": {
                "type": "object",
                "properties": {"notas": {"type": "array", "items": {"type": "number"}, "description": "Lista de notas."}},
                "required": ["notas"],
            },
        },
    },
]

print(sorted(AVAILABLE_TOOLS))

## 3. Um passo só não basta

Teste o fluxo da Aula 3 com uma pergunta que precisa de duas ferramentas encadeadas. A segunda ferramenta só pode ser escolhida depois que o resultado da primeira for conhecido.

In [ ]:
query = "Estou aprovado em banco_de_dados?"
messages = [
    {"role": "system", "content": "Você é um assistente de estudos. Use ferramentas quando precisar de dados ou cálculos."},
    {"role": "user", "content": query},
]

# TODO: faça a primeira chamada com `tools` e `tool_choice="auto"`.
# Guarde em `first_response` e imprima os tool_calls da mensagem.
# Pergunta a responder: o modelo conseguiu pedir todas as ferramentas necessárias de uma vez?

## 4. Execute uma chamada de ferramenta com segurança

No loop, a mesma lógica da Aula 3 será repetida várias vezes, então vale isolá-la em uma função. Há uma mudança importante: quando algo dá errado (ferramenta desconhecida, JSON inválido, argumento fora do intervalo), a função **não interrompe o programa**. Ela devolve o erro como resultado, para o modelo ler e tentar de novo.

O nome vindo do modelo continua sem virar código arbitrário: só funções do mapa `AVAILABLE_TOOLS` podem ser executadas.

In [ ]:
# TODO: implemente `executar_tool_call(call)`.
# 1. leia call.function.name e call.function.arguments;
# 2. se o nome não estiver em AVAILABLE_TOOLS, retorne {"error": "Ferramenta não permitida: ..."};
# 3. use json.loads(arguments) dentro de try/except e retorne {"error": ...} se falhar;
# 4. execute AVAILABLE_TOOLS[name](**arguments) dentro de try/except;
#    em caso de exceção, retorne {"error": str(exc)};
# 5. retorne o resultado da função.

## 5. Implemente o loop do agente

O agente é o mesmo ciclo da Aula 3, repetido. A cada iteração:

1. a aplicação chama o modelo com o histórico completo;
2. a mensagem do modelo entra no histórico;
3. se **não** há `tool_calls`, o modelo terminou e essa é a resposta final;
4. se há `tool_calls`, a aplicação executa cada uma e acrescenta uma mensagem `tool` por resultado;
5. o ciclo recomeça.

Repare que quem decide quando parar é o modelo. Quem impõe o limite de segurança é a aplicação.

In [ ]:
# TODO: implemente `run_agent(query, max_steps=5)`.
# 1. crie `messages` com a mensagem system e a pergunta do usuário;
# 2. use `for step in range(1, max_steps + 1)`;
# 3. em cada passo, chame o modelo com tools e tool_choice="auto";
# 4. acrescente a mensagem do modelo ao histórico;
# 5. se não houver tool_calls, retorne a resposta final e o número de passos;
# 6. caso contrário, execute cada chamada com `executar_tool_call`
#    e acrescente uma mensagem tool com o tool_call_id correto;
# 7. se o for terminar sem resposta final, retorne uma indicação de que o limite foi atingido.

## 6. Registre cada passo

Um agente que só mostra a resposta final é difícil de depurar. Acrescente um traço legível: em cada passo, imprima qual ferramenta foi pedida, com quais argumentos e o que voltou. É esse registro que mostra, por exemplo, quando o modelo repete a mesma chamada ou insiste em um argumento errado.

In [ ]:
# TODO: altere `run_agent` para receber `verbose=True` e imprimir, em cada chamada de ferramenta:
# [passo 1] consultar_notas({"disciplina": "banco_de_dados"}) -> {...}
# e, no fim, a resposta final e o total de passos.

## 7. Teste o limite de iterações

Sem um limite, um erro de prompt ou de ferramenta pode manter o agente pedindo a mesma coisa para sempre, gastando tokens e dinheiro. Reduza `max_steps` para um valor menor que o necessário e veja o que acontece.

In [ ]:
# TODO: execute a mesma pergunta com max_steps=1.
# Observe: result["answer"] é None? result["stopped"] indica o limite?
# Pense: o que a aplicação deveria mostrar ao usuário quando o limite é atingido?

## 8. Veja o agente lidar com erros

Como os erros voltam ao modelo como resultado, ele pode tentar corrigir o próprio pedido. Provoque isso com perguntas que levam a falhas.

In [ ]:
# TODO: teste as perguntas abaixo com o traço ligado e registre o que o agente fez.
# 1. "Estou aprovado em cálculo?"  -> a disciplina não existe; o que a ferramenta devolve?
# 2. "Qual o resumo da aula 9?"    -> a aula não existe; o modelo inventa ou admite?
# 3. "Qual a média de 11, 7 e 9?"  -> a nota 11 é inválida; como o erro chega ao modelo?

## 9. Compare os caminhos

Agora você tem três comportamentos possíveis. Faça uma pergunta de cada tipo e registre o que aconteceu.

In [ ]:
# TODO: preencha a tabela com o que observou.
#
# | Pergunta                                             | Caminho             | Passos | Ferramentas usadas |
# |------------------------------------------------------|---------------------|--------|--------------------|
# | "O que é um agente?"                                 | resposta direta     |        |                    |
# | "Qual é o tema da aula 3?"                           | uma ferramenta      |        |                    |
# | "Qual aula fala de RAG e o que ela cobre?"           | várias ferramentas  |        |                    |
# | "Estou aprovado em banco_de_dados?"                  | várias ferramentas  |        |                    |

## 10. Checkpoint da V3.0

A V3.0 está pronta quando:

- as ferramentas estão descritas em `tools` e mapeadas em `AVAILABLE_TOOLS`;
- `executar_tool_call` valida o nome e devolve erros como resultado;
- `run_agent` repete o ciclo até o modelo responder sem `tool_calls`;
- cada resultado usa o `tool_call_id` correto;
- existe um limite de iterações e o caso de limite atingido é tratado;
- o traço mostra cada passo, ferramenta e resultado;
- a chave continua no `.env`;
- as alterações foram salvas em um commit separado.

## 11. Próxima aula

Na Aula 6, o assistente ganha acesso a uma base de documentos com **RAG**: em vez de depender só das ferramentas que temos, ele recupera trechos relevantes antes de responder. A busca poderá ser exposta como mais uma ferramenta deste agente.